# KNN Coffee Bean Predictor
Predicts a coffee bean (`name`) from roast level, aroma, structure, body, flavour, and aftertaste.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')

In [18]:
df = pd.read_csv('coffee_data.csv')
df['agtron_whole'] = df['agtron'].str.extract(r'^(\d+)').astype(float)
print(f'Shape: {df.shape}')
print(f'Roast levels: {df["roast_level"].unique()}')
print('Missing values:')
print(df.isnull().sum())
df.head()

Shape: (9000, 14)
Roast levels: ['Medium-Dark' 'Medium' 'Medium-Light' 'Light' 'Dark' 'Very Dark' nan]
Missing values:
name                   0
roaster                0
score                  4
roast_level          405
aroma                126
structure           5085
body                  68
flavour               72
aftertaste           847
coffee_origin        484
roaster_location       4
agtron                 0
cost                2012
agtron_whole         311
dtype: int64


,name,roaster,score,roast_level,aroma,structure,body,flavour,aftertaste,coffee_origin,roaster_location,agtron,cost,agtron_whole
0,Ka’u Classic Dark Roast,Rusty's Hawaiian,95.0,Medium-Dark,9.0,9.0,9.0,9.0,9.0,"Ka‘ū growing district, Hawai'i Island, Hawai'i","Pahala, Hawai'i Island, Hawai'i",44/64,$22.50/7 ounces,44.0
1,Kenya Peaberry Hera,Simon Hsieh Aroma Roast Coffees,95.0,Medium-Dark,9.0,9.0,9.0,9.0,9.0,"Nyeri growing region, south-central Kenya","Taiyuan, Taiwan",44/62,"NT $1,100/227 grams",44.0
2,Ethiopia Washed Limu,Kakalove Cafe,94.0,Medium-Dark,9.0,9.0,9.0,9.0,8.0,"Guji Zone, Oromia Region, southern Ethiopia","Chia-Yi, Taiwan",47/64,NT $290/8 ounces,47.0
3,Strawberry Shadow,modcup,94.0,Medium-Dark,9.0,8.0,9.0,9.0,9.0,"Huila and Quindío Departments, Colombia","Jersey City, New Jersey",46/63,$30.00/250 grams,46.0
4,Ethiopia Uraga Perfume Rose Anaerobic Washed,Linsun Coffee,93.0,Medium-Dark,9.0,8.0,9.0,9.0,8.0,"Guji Zone, Oromia Region, south-central Ethiopia","Tamsui, Taiwan",45/71,NT $800/200 grams,45.0


In [ ]:
FEATURES = [
    'roast_level',
    'agtron_whole',
    'aroma', 'structure', 'body', 'flavour', 'aftertaste',
    'score',
    'coffee_origin',
]

roast_order = [['Light', 'Medium-Light', 'Medium', 'Medium-Dark', 'Dark']]

preprocessor = ColumnTransformer(transformers=[
    ('roast', Pipeline([
        ('encode', OrdinalEncoder(categories=roast_order, handle_unknown='use_encoded_value', unknown_value=-1)),
        ('scale', StandardScaler())
    ]), ['roast_level']),
    ('numeric', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), ['agtron_whole', 'aroma', 'structure', 'body', 'flavour', 'aftertaste', 'score']),
    ('origin', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), ['coffee_origin'])
])

print('Preprocessor ready.')

In [ ]:
# Step 2 & 3: transform every coffee into a feature vector and store
coffee_vectors = preprocessor.fit_transform(df[FEATURES])
X = coffee_vectors
print(f'Feature matrix: {X.shape}  ({len(df)} coffees × {X.shape[1]} features)')

In [ ]:
knn = NearestNeighbors(metric='cosine')
knn.fit(X)
print('KNN fitted on full dataset.')

In [ ]:
def recommend_coffee(
        roast_level, aroma, structure, body, flavour, aftertaste,
        score=95, agtron_whole=None, coffee_origin=None,
        top_n=5):
    user_input = pd.DataFrame([{
        'roast_level':   roast_level,
        'agtron_whole':  agtron_whole,
        'aroma':         aroma,
        'structure':     structure,
        'body':          body,
        'flavour':       flavour,
        'aftertaste':    aftertaste,
        'score':         score,
        'coffee_origin': coffee_origin,
    }])

    # Step 4: transform user input at prediction time
    query = preprocessor.transform(user_input)

    # Step 5: find nearest neighbours using cosine distance
    distances, indices = knn.kneighbors(query, n_neighbors=top_n)
    similarities = 1 - distances[0]  # cosine similarity = 1 - cosine distance

    # Step 6 & 7: return top N with confidence
    names = df['name'].values
    print(f'Top {top_n} recommended coffees:')
    for rank, (idx, sim) in enumerate(zip(indices[0], similarities), 1):
        print(f'  {rank}. {names[idx]:<55s} {sim:.2f}')

print('recommend_coffee() ready.')

In [ ]:
recommend_coffee('Light', aroma=9, structure=9, body=9, flavour=10, aftertaste=9)

## V5 Sample Output

Query: `roast=Light, aroma=9, structure=9, body=9, flavour=10, aftertaste=9`

```
Top 20 recommended coffees:
  1. Kona S12 Kaffa Washed                                   0.95
  2. Colombia Cauca El Paraiso                               0.94
  3. Yemen Lot 106                                           0.92
  4. Panama Elida Natural Lot #13                            0.91
  5. David Mburu Kenya                                       0.91
  6. Kona Orange                                             0.91
  7. Kenya Thageini                                          0.91
  8. Yemen Microlot                                          0.91
  9. Wilton Benitez Java                                     0.91
  10. Hawaii Kilauea Volcano Nano-Lot                         0.91
  11. Dodora Double                                           0.91
  12. Yemen Haraaz                                            0.91
  13. Wilton Benitez Geisha                                   0.91
  14. Colombia Villa Betulia CM Sidra                         0.91
  15. Colombia Triple-Anaerobic Honey El Triunfo Pink Bourbon 0.91
  16. Guatemala Washed El Injerto EI05 Mocca                  0.91
  17. Ethiopia Banko Washed                                   0.91
  18. Ecuador COE 1st place Arashi Typica Mejorado Washed     0.91
  19. Guatemala Santa Felisa Wild Yeast Natural Gesha         0.91
  20. Ethiopia Natural Sidama Tuke Fengshui G1                0.91
```

**Notes:**
- Top 4 show meaningful differentiation (0.95 → 0.91)
- Positions 5–20 all tie at 0.91 — ranking within this band is arbitrary (dataset order), not by similarity
- Tie collapse is expected: tasting scores are discrete integers, so many coffees map to nearly identical vectors in the normalised feature space
- KNN retrieves exact nearest neighbours; the cosine model's `argsort` over all vectors produces the same top-4 with the same scores — both models agree on the confident recommendations
- For top-5 use, the tie issue doesn't matter; it only surfaces when requesting more results